# Track2Sheet: song → stems

1. **Download** the song from YouTube with [yt-dlp](https://github.com/yt-dlp/yt-dlp) → `output/downloads/<song>.wav`
2. **Separate** it into 6 stems (vocals, drums, bass, guitar, piano, other) with **BS-RoFormer SW**
   via [audio-separator](https://github.com/nomadkaraoke/python-audio-separator) → `output/stems/<song>/<stem>.wav`

Settings come from `.env` in the project root. Steps whose output already exists are skipped;
set `FORCE = True` to redo them.

## Setup

In [ ]:
import base64
import logging
import os
import re
import shutil
import subprocess
import tempfile
from pathlib import Path

import torch
import yt_dlp
from dotenv import find_dotenv, load_dotenv
from IPython.display import HTML, display

env_file = find_dotenv(usecwd=True)
load_dotenv(env_file)
ROOT = Path(env_file).parent if env_file else Path.cwd().parent

URL = os.getenv("YOUTUBE_URL", "")          # or paste a link here
OUTPUT_DIR = ROOT / os.getenv("OUTPUT_DIR", "output")
MODEL_DIR = ROOT / os.getenv("MODEL_DIR", "models")
SEPARATION_MODEL = os.getenv("SEPARATION_MODEL", "BS-Roformer-SW.ckpt")
STEM_FORMAT = os.getenv("STEM_FORMAT", "WAV")
COOKIES = os.getenv("YTDLP_COOKIES") or None
FORCE = False

device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (no CUDA GPU found)"
print(f"project : {ROOT}")
print(f"song    : {URL}")
print(f"model   : {SEPARATION_MODEL}")
print(f"device  : {device}")

In [ ]:
def player(label: str, path: Path) -> str:
    """Audio player with the file embedded as a small MP3 (works in any Jupyter frontend)."""
    with tempfile.TemporaryDirectory() as tmp:
        mp3 = Path(tmp) / "a.mp3"
        subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", str(path), "-b:a", "96k", str(mp3)],
                       check=True)
        b64 = base64.b64encode(mp3.read_bytes()).decode()
    return (f'<div style="display:flex;align-items:center;gap:12px;margin:2px 0">'
            f'<code style="width:90px">{label}</code><audio controls preload="none" '
            f'style="width:420px" src="data:audio/mpeg;base64,{b64}"></audio></div>')

## 1. Download

In [ ]:
def download_audio(url: str, out_dir: Path) -> Path:
    """Download the best audio stream as WAV; reuse it if it's already there."""
    out_dir.mkdir(parents=True, exist_ok=True)
    opts = {
        "format": "bestaudio/best",
        "outtmpl": str(out_dir / "%(title)s [%(id)s].%(ext)s"),
        "restrictfilenames": True,
        "noplaylist": True,
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav"}],
    }
    if COOKIES:
        opts["cookiefile"] = str(ROOT / COOKIES)
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(url, download=False)
        wav = Path(ydl.prepare_filename(info)).with_suffix(".wav")
        if wav.exists() and not FORCE:
            print(f"Already downloaded: {wav.name}")
            return wav
        info = ydl.process_ie_result(info, download=True)
        return Path(info["requested_downloads"][0]["filepath"])


mix = download_audio(URL, OUTPUT_DIR / "downloads")
SONG = mix.stem
display(HTML(player("mix", mix)))

## 2. Separate into stems (BS-RoFormer)

In [ ]:
def separate(audio: Path, out_dir: Path) -> list[Path]:
    """Split `audio` into stems in `out_dir`, named <stem>.<ext>; reuse existing stems."""
    existing = sorted(out_dir.glob(f"*.{STEM_FORMAT.lower()}"))
    if existing and not FORCE:
        print(f"Stems already in {out_dir}")
        return existing

    from audio_separator.separator import Separator

    out_dir.mkdir(parents=True, exist_ok=True)
    separator = Separator(output_dir=str(out_dir), output_format=STEM_FORMAT,
                          model_file_dir=str(MODEL_DIR), log_level=logging.WARNING)
    separator.load_model(model_filename=SEPARATION_MODEL)
    print(f"Separating on {device}...")
    outputs = separator.separate(str(audio))

    # "<song>_(Vocals)_BS-Roformer-SW.wav" -> "vocals.wav"
    stems = []
    for f in outputs:
        src = Path(f) if Path(f).is_absolute() else out_dir / f
        names = re.findall(r"_\(([^)]+)\)", src.stem)
        dst = out_dir / f"{(names[-1] if names else src.stem).lower()}{src.suffix}"
        shutil.move(src, dst)
        stems.append(dst)

    del separator
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return sorted(stems)


stems = separate(mix, OUTPUT_DIR / "stems" / SONG)
display(HTML("".join(player(p.stem, p) for p in stems)))